In [1]:
import numpy as np
import cv2
import onnxruntime as ort
import time
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union
from collections import defaultdict
import warnings

# ============================================
# COCO 80 classes
# ============================================
COCO_NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

# ============================================
# Pascal VOC 20 classes
# ============================================
VOC_NAMES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 
    'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 
    'sheep', 'sofa', 'train', 'tvmonitor'
]

# ============================================
# COCO -> VOC mapping (только 20 пересекающихся классов)
# ============================================
COCO_TO_VOC_MAPPING = {
    'person': 'person',
    'bird': 'bird',
    'cat': 'cat',
    'cow': 'cow',
    'dog': 'dog',
    'horse': 'horse',
    'sheep': 'sheep',
    'airplane': 'aeroplane',
    'bicycle': 'bicycle',
    'boat': 'boat',
    'bus': 'bus',
    'car': 'car',
    'motorcycle': 'motorbike',
    'train': 'train',
    'bottle': 'bottle',
    'chair': 'chair',
    'dining table': 'diningtable',
    'potted plant': 'pottedplant',
    'couch': 'sofa',
    'tv': 'tvmonitor'
}

# Создаём индексы для быстрого доступа
COCO_TO_VOC_INDEX = {}
VOC_NAME_TO_INDEX = {name: idx for idx, name in enumerate(VOC_NAMES)}

for coco_name, voc_name in COCO_TO_VOC_MAPPING.items():
    coco_idx = COCO_NAMES.index(coco_name)
    voc_idx = VOC_NAMES.index(voc_name)
    COCO_TO_VOC_INDEX[coco_idx] = voc_idx


class YOLODetector:
    """
    Unified detector for YOLOX-Tiny and YOLOv5-nano ONNX models.
    Поддерживает оценку на Pascal VOC с автоматическим маппингом классов.
    """
    
    def __init__(self, model_path: str, model_type: str = 'yolox', input_size: int = 416,
                 conf_thres: float = 0.3, nms_thres: float = 0.45, 
                 providers: Optional[List[str]] = None):
        self.model_path = model_path
        self.model_type = model_type.lower()
        self.input_size = input_size
        self.conf_thres = conf_thres
        self.nms_thres = nms_thres
        self.num_classes = 80  # COCO classes
        
        if providers is None:
            providers = ['CoreMLExecutionProvider']
        
        self.session = ort.InferenceSession(model_path, providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        
        self.strides = [8, 16, 32]
        self.anchors = [
            [[10, 13], [16, 30], [33, 23]],
            [[30, 61], [62, 45], [59, 119]],
            [[116, 90], [156, 198], [373, 326]]
        ]
        
        # YOLOv5 format detection
        self.yolov5_format = None
        if self.model_type == 'yolov5':
            outputs = self.session.get_outputs()
            if len(outputs) == 3:
                self.yolov5_format = 'multi'
            elif len(outputs) == 1:
                self.yolov5_format = 'single'
            else:
                raise ValueError(f"YOLOv5: неподдерживаемое количество выходов: {len(outputs)}")
            print(f"[INFO] YOLOv5 формат выхода: {self.yolov5_format}")
        
        self._grids = None
        self._expanded_strides = None
    
    def preprocess(self, image: Union[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray, tuple]:
        if isinstance(image, str):
            img = cv2.imread(image)
            if img is None:
                raise ValueError(f"Не удалось загрузить {image}")
        else:
            img = image.copy()
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_shape = img.shape[:2]
        h, w = original_shape
        
        scale = min(self.input_size / h, self.input_size / w)
        new_h, new_w = int(h * scale), int(w * scale)
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        padded = np.full((self.input_size, self.input_size, 3), 114, dtype=np.uint8)
        padded[:new_h, :new_w] = resized
        
        padded = padded.astype(np.float32)
        if self.model_type == 'yolov5':
            padded = padded / 255.0
        
        padded = padded.transpose(2, 0, 1)
        input_tensor = np.expand_dims(padded, 0)
        
        return input_tensor, img, (scale, new_h, new_w)
    
    def _make_grid(self, nx: int, ny: int) -> np.ndarray:
        xv, yv = np.meshgrid(np.arange(nx), np.arange(ny))
        return np.stack((xv, yv), axis=2).reshape((1, 1, ny, nx, 2)).astype(np.float32)
    
    def _get_yolox_grids(self):
        if self._grids is None:
            grids = []
            expanded_strides = []
            for stride in self.strides:
                hsize = self.input_size // stride
                wsize = self.input_size // stride
                yv, xv = np.meshgrid(np.arange(hsize), np.arange(wsize), indexing='ij')
                grid = np.stack((xv, yv), axis=2).reshape(-1, 2)
                grids.append(grid)
                expanded_strides.append(np.full((grid.shape[0], 1), stride))
            
            self._grids = np.concatenate(grids, axis=0)
            self._expanded_strides = np.concatenate(expanded_strides, axis=0)
        
        return self._grids, self._expanded_strides
    
    def decode(self, outputs: List[np.ndarray]) -> np.ndarray:
        if self.model_type == 'yolox':
            return self._decode_yolox(outputs[0])
        elif self.model_type == 'yolov5':
            if self.yolov5_format == 'multi':
                return self._decode_yolov5_multi(outputs)
            else:
                return self._decode_yolov5_single(outputs[0])
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
    
    def _decode_yolox(self, raw_output: np.ndarray) -> np.ndarray:
        predictions = raw_output[0]
        grids, expanded_strides = self._get_yolox_grids()
        
        box_preds = predictions[:, :4]
        obj_preds = predictions[:, 4:5]
        cls_preds = predictions[:, 5:]
        
        x_center = (grids[:, 0:1] + box_preds[:, 0:1]) * expanded_strides
        y_center = (grids[:, 1:2] + box_preds[:, 1:2]) * expanded_strides
        w = np.exp(box_preds[:, 2:3]) * expanded_strides
        h = np.exp(box_preds[:, 3:4]) * expanded_strides
        
        x1 = x_center - w / 2
        y1 = y_center - h / 2
        x2 = x_center + w / 2
        y2 = y_center + h / 2
        
        decoded = np.concatenate([x1, y1, x2, y2, obj_preds, cls_preds], axis=1)
        return decoded
    
    def _decode_yolov5_multi(self, outputs: List[np.ndarray]) -> np.ndarray:
        def sigmoid(x):
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        
        all_predictions = []
        
        for i, (stride, anchor, pred) in enumerate(zip(self.strides, self.anchors, outputs)):
            batch_size, num_anchors, ny, nx, features = pred.shape
            pred = sigmoid(pred)
            
            grid = self._make_grid(nx, ny)
            anchor_grid = np.array(anchor).reshape((1, num_anchors, 1, 1, 2))
            
            pred[..., 0:2] = (pred[..., 0:2] * 2. - 0.5 + grid) * stride
            pred[..., 2:4] = (pred[..., 2:4] * 2) ** 2 * anchor_grid
            
            pred = pred.reshape((batch_size, -1, features))
            all_predictions.append(pred)
        
        predictions = np.concatenate(all_predictions, axis=1)[0]
        
        boxes = predictions[:, :4]
        xyxy = np.copy(boxes)
        xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2
        xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2
        xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2
        xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2
        predictions[:, :4] = xyxy
        
        return predictions
    
    def _decode_yolov5_single(self, raw_output: np.ndarray) -> np.ndarray:
        predictions = raw_output[0]
        predictions = predictions.T
        
        boxes = predictions[:, :4]
        class_scores = predictions[:, 4:]
        
        objness = np.max(class_scores, axis=1, keepdims=True)
        
        if boxes.max() <= 1.0:
            boxes = boxes * self.input_size
        
        xyxy = np.copy(boxes)
        xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2
        xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2
        xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2
        xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2
        
        decoded = np.concatenate([xyxy, objness, class_scores], axis=1)
        return decoded
    
    def postprocess(self, predictions: np.ndarray, scale_info: tuple, 
                    filter_voc_only: bool = False) -> np.ndarray:
        """
        Постобработка с опциональной фильтрацией только VOC-классов.
        """
        scale, new_h, new_w = scale_info
        
        boxes = predictions[:, :4]
        objness = predictions[:, 4]
        class_scores = predictions[:, 5:]
        
        scores = objness[:, np.newaxis] * class_scores
        max_scores = np.max(scores, axis=1)
        class_ids = np.argmax(scores, axis=1)
        
        # Фильтрация по confidence
        valid_mask = max_scores > self.conf_thres
        boxes = boxes[valid_mask]
        scores = max_scores[valid_mask]
        class_ids = class_ids[valid_mask]
        
        if len(boxes) == 0:
            return np.array([])
        
        # Фильтрация только VOC классов (если нужно)
        if filter_voc_only:
            voc_mask = np.array([cls_id in COCO_TO_VOC_INDEX for cls_id in class_ids])
            boxes = boxes[voc_mask]
            scores = scores[voc_mask]
            class_ids = class_ids[voc_mask]
            
            if len(boxes) == 0:
                return np.array([])
        
        # NMS
        indices = cv2.dnn.NMSBoxes(
            boxes.tolist(),
            scores.tolist(),
            self.conf_thres,
            self.nms_thres
        )
        
        if len(indices) == 0:
            return np.array([])
        
        indices = indices.flatten() if hasattr(indices, 'flatten') else indices
        
        dets = []
        for idx in indices:
            dets.append([*boxes[idx], scores[idx], class_ids[idx]])
        dets = np.array(dets)
        
        # Масштабирование к оригинальному размеру
        dets[:, [0, 2]] = dets[:, [0, 2]] / scale
        dets[:, [1, 3]] = dets[:, [1, 3]] / scale
        
        return dets
    
    def predict(self, image: Union[str, np.ndarray], filter_voc_only: bool = False) -> Tuple[np.ndarray, np.ndarray]:
        """
        Предсказание с опцией фильтрации только VOC классов.
        """
        input_tensor, original_img, scale_info = self.preprocess(image)
        outputs = self.session.run(None, {self.input_name: input_tensor})
        predictions = self.decode(outputs)
        detections = self.postprocess(predictions, scale_info, filter_voc_only=filter_voc_only)
        return detections, original_img
    
    def parse_voc_annotation(self, xml_path: str) -> List[Tuple]:
        """
        Парсинг XML аннотации Pascal VOC.
        Returns: [(class_id, x1, y1, x2, y2), ...] в VOC индексах (0-19)
        """
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        objects = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            
            # Игнорируем difficult объекты (опционально)
            difficult = obj.find('difficult')
            if difficult is not None and int(difficult.text) == 1:
                continue
            
            if name not in VOC_NAMES:
                continue
            
            bbox = obj.find('bndbox')
            x1 = float(bbox.find('xmin').text)
            y1 = float(bbox.find('ymin').text)
            x2 = float(bbox.find('xmax').text)
            y2 = float(bbox.find('ymax').text)
            
            class_id = VOC_NAMES.index(name)
            objects.append((class_id, x1, y1, x2, y2))
        
        return objects
    
    def evaluate_voc(self, 
                     voc_root: str, 
                     image_set: str = 'val',
                     voc_year: str = '2012',
                     iou_thresh: float = 0.5) -> Dict:
        """
        Оценка модели на Pascal VOC датасете.
        
        Args:
            voc_root: Путь к корню VOC (где лежат JPEGImages, Annotations и т.д.)
            image_set: 'train', 'val', 'trainval', 'test'
            voc_year: '2007' или '2012'
            iou_thresh: Порог IoU для mAP
        
        Returns:
            Результаты оценки: mAP, AP per class, precision/recall
        """
        voc_root = Path(voc_root)
        
        # Читаем список изображений
        list_file = voc_root / 'ImageSets' / 'Main' / f'{image_set}.txt'
        if not list_file.exists():
            # Пробуем с годом: trainval_2012
            list_file = voc_root / 'ImageSets' / 'Main' / f'{image_set}_{voc_year}.txt'
        
        if not list_file.exists():
            raise FileNotFoundError(f"Не найден список изображений: {list_file}")
        
        with open(list_file, 'r') as f:
            image_ids = [line.strip().split()[0] for line in f.readlines()]
        
        print(f"\n{'='*60}")
        print(f"VOC {voc_year} {image_set} Evaluation")
        print(f"{'='*60}")
        print(f"Всего изображений: {len(image_ids)}")
        print(f"Модель: {self.model_type.upper()}")
        print(f"IoU threshold: {iou_thresh}")
        print(f"{'='*60}\n")
        
        # Собираем GT и предсказания по классам
        gt_by_class = defaultdict(list)  # class_id -> [{'image_id': ..., 'bbox': ..., 'used': False}]
        preds_by_class = defaultdict(list)  # class_id -> [{'image_id': ..., 'bbox': ..., 'score': ...}]
        
        total_inference_time = 0
        
        for idx, img_id in enumerate(image_ids):
            if idx % 100 == 0:
                print(f"Processing [{idx}/{len(image_ids)}] {img_id}...")
            
            # Пути к файлам
            img_path = voc_root / 'JPEGImages' / f'{img_id}.jpg'
            xml_path = voc_root / 'Annotations' / f'{img_id}.xml'
            
            if not img_path.exists() or not xml_path.exists():
                continue
            
            # Загружаем GT
            gt_objects = self.parse_voc_annotation(str(xml_path))
            for class_id, x1, y1, x2, y2 in gt_objects:
                gt_by_class[class_id].append({
                    'image_id': img_id,
                    'bbox': [x1, y1, x2, y2],
                    'used': False
                })
            
            # Инференс (фильтруем только VOC классы и конвертируем индексы)
            start = time.perf_counter()
            detections, _ = self.predict(str(img_path), filter_voc_only=True)
            total_inference_time += (time.perf_counter() - start)
            
            # Конвертируем COCO индексы в VOC индексы
            for det in detections:
                x1, y1, x2, y2, score, coco_cls_id = det
                coco_cls_id = int(coco_cls_id)
                
                if coco_cls_id in COCO_TO_VOC_INDEX:
                    voc_cls_id = COCO_TO_VOC_INDEX[coco_cls_id]
                    preds_by_class[voc_cls_id].append({
                        'image_id': img_id,
                        'bbox': [x1, y1, x2, y2],
                        'score': float(score)
                    })
        
        # Вычисляем AP для каждого класса VOC
        aps = {}
        class_precisions = {}
        class_recalls = {}
        
        print(f"\n{'─'*60}")
        print("Per-class AP:")
        print(f"{'─'*60}")
        
        for class_id in range(len(VOC_NAMES)):
            class_name = VOC_NAMES[class_id]
            preds = preds_by_class[class_id]
            gts = gt_by_class[class_id]
            
            num_gt = len(gts)
            if num_gt == 0:
                continue
            
            # Сортируем по score (убывание)
            preds = sorted(preds, key=lambda x: x['score'], reverse=True)
            
            tp = np.zeros(len(preds))
            fp = np.zeros(len(preds))
            
            # Для каждого предсказания ищем лучший match
            for pred_idx, pred in enumerate(preds):
                pred_bbox = pred['bbox']
                img_id = pred['image_id']
                
                # Ищем соответствующий GT на этом изображении
                best_iou = 0
                best_gt_idx = -1
                
                for gt_idx, gt in enumerate(gts):
                    if gt['image_id'] != img_id or gt['used']:
                        continue
                    
                    iou = self._compute_iou(pred_bbox, gt['bbox'])
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx
                
                if best_iou >= iou_thresh and best_gt_idx != -1:
                    tp[pred_idx] = 1
                    gts[best_gt_idx]['used'] = True
                else:
                    fp[pred_idx] = 1
            
            # Вычисляем precision/recall
            tp_cumsum = np.cumsum(tp)
            fp_cumsum = np.cumsum(fp)
            
            recalls = tp_cumsum / num_gt
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-16)
            
            # 11-point interpolation (VOC стиль)
            ap = self._voc_ap(recalls, precisions)
            aps[class_id] = ap
            
            # Сохраняем финальные P/R для отчёта
            if len(precisions) > 0:
                class_precisions[class_name] = float(precisions[-1])
                class_recalls[class_name] = float(recalls[-1])
            else:
                class_precisions[class_name] = 0.0
                class_recalls[class_name] = 0.0
            
            print(f"{class_name:<15} AP: {ap:.4f}  ({int(np.sum(tp))}/{num_gt} objects detected)")
        
        # mAP
        mean_ap = np.mean(list(aps.values())) if len(aps) > 0 else 0.0
        avg_time = total_inference_time / len(image_ids) * 1000  # ms
        fps = 1000 / avg_time if avg_time > 0 else 0
        
        results = {
            'model_type': self.model_type,
            'model_path': self.model_path,
            'voc_year': voc_year,
            'image_set': image_set,
            'num_images': len(image_ids),
            'mAP@0.5': float(mean_ap),
            'fps': float(fps),
            'avg_inference_time_ms': float(avg_time),
            'per_class_ap': {VOC_NAMES[k]: float(v) for k, v in aps.items()},
            'per_class_precision': class_precisions,
            'per_class_recall': class_recalls
        }
        
        print(f"\n{'='*60}")
        print("RESULTS SUMMARY")
        print(f"{'='*60}")
        print(f"mAP@0.5: {mean_ap:.4f} ({mean_ap*100:.2f}%)")
        print(f"FPS:     {fps:.2f}")
        print(f"Latency: {avg_time:.2f} ms")
        print(f"{'='*60}\n")
        
        return results
    
    def _compute_iou(self, box1: List[float], box2: List[float]) -> float:
        """IoU между двумя боксами [x1, y1, x2, y2]"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        union = area1 + area2 - inter
        return inter / (union + 1e-16) if union > 0 else 0
    
    def _voc_ap(self, recalls: np.ndarray, precisions: np.ndarray) -> float:
        """
        11-point interpolation AP (VOC style).
        """
        # Добавляем начальную и конечную точки
        recalls = np.concatenate(([0.], recalls, [1.]))
        precisions = np.concatenate(([0.], precisions, [0.]))
        
        # Делаем precision монотонно убывающей
        for i in range(len(precisions) - 2, -1, -1):
            precisions[i] = max(precisions[i], precisions[i + 1])
        
        # 11-point interpolation
        ap = 0.0
        for t in np.arange(0, 1.1, 0.1):
            if np.sum(recalls >= t) == 0:
                p = 0
            else:
                p = np.max(precisions[recalls >= t])
            ap += p / 11.0
        
        return ap
    
    def benchmark_speed(self, num_runs: int = 100, warmup: int = 10) -> Dict:
        """Замер скорости на dummy данных."""
        dummy_input = np.random.randn(1, 3, self.input_size, self.input_size).astype(np.float32)
        if self.model_type == 'yolov5':
            dummy_input = dummy_input / 255.0
        
        for _ in range(warmup):
            self.session.run(None, {self.input_name: dummy_input})
        
        times = []
        for _ in range(num_runs):
            start = time.perf_counter()
            self.session.run(None, {self.input_name: dummy_input})
            end = time.perf_counter()
            times.append((end - start) * 1000)
        
        times = np.array(times)
        return {
            'fps': 1000.0 / times.mean(),
            'mean_latency_ms': times.mean(),
            'std_latency_ms': times.std()
        }
    
    def calculate_flops(self) -> Dict:
        """Подсчёт FLOPS через ONNX."""
        try:
            import onnx
        except ImportError:
            return {'flops_g': 0, 'params_m': 0}
        
        model = onnx.load(self.model_path)
        graph = model.graph
        
        total_flops = 0
        total_params = 0
        
        value_info = {vi.name: vi for vi in list(graph.value_info) + list(graph.input) + list(graph.output)}
        
        def get_shape(tensor_name):
            if tensor_name in value_info:
                tensor = value_info[tensor_name]
                return [d.dim_value for d in tensor.type.tensor_type.shape.dim]
            return None
        
        for node in graph.node:
            if node.op_type == 'Conv':
                inputs = list(node.input)
                if len(inputs) < 2:
                    continue
                
                attrs = {attr.name: attr for attr in node.attribute}
                kernel_shape = attrs.get('kernel_shape')
                if kernel_shape:
                    k_h, k_w = kernel_shape.ints[0], kernel_shape.ints[1]
                else:
                    k_h = k_w = 3
                
                input_shape = get_shape(inputs[0])
                output_shape = get_shape(node.output[0])
                
                if input_shape and output_shape and len(input_shape) >= 4:
                    in_c = input_shape[1]
                    out_c = output_shape[1]
                    out_h = output_shape[2]
                    out_w = output_shape[3]
                    
                    flops = 2 * in_c * out_c * k_h * k_w * out_h * out_w
                    total_flops += flops
                    
                    params = out_c * in_c * k_h * k_w
                    if len(inputs) > 2:
                        params += out_c
                    total_params += params
        
        return {
            'flops_g': total_flops / 1e9,
            'params_m': total_params / 1e6
        }


def compare_models_on_voc(voc_root: str, 
                          yolox_path: str, 
                          yolov5_path: str,
                          output_json: str = "voc_comparison.json",
                          image_set: str = 'val',
                          voc_year: str = '2012'):
    """
    Сравнение YOLOX и YOLOv5 на VOC датасете с выводом таблицы результатов.
    """
    print("\n" + "="*80)
    print("PASCAL VOC EVALUATION: YOLOX vs YOLOv5")
    print("="*80)
    print(f"Dataset: VOC {voc_year} {image_set}")
    print(f"COCO->VOC mapping: {len(COCO_TO_VOC_MAPPING)} classes")
    print("="*80 + "\n")
    
    results = {}
    
    # YOLOX
    print("[1/2] Evaluating YOLOX...")
    detector_x = YOLODetector(yolox_path, model_type='yolox', conf_thres=0.3)
    results['yolox'] = detector_x.evaluate_voc(voc_root, image_set, voc_year)
    flops_x = detector_x.calculate_flops()
    results['yolox']['flops_g'] = flops_x['flops_g']
    results['yolox']['params_m'] = flops_x['params_m']
    
    # YOLOv5
    print("[2/2] Evaluating YOLOv5...")
    detector_v5 = YOLODetector(yolov5_path, model_type='yolov5', conf_thres=0.3)
    results['yolov5'] = detector_v5.evaluate_voc(voc_root, image_set, voc_year)
    flops_v5 = detector_v5.calculate_flops()
    results['yolov5']['flops_g'] = flops_v5['flops_g']
    results['yolov5']['params_m'] = flops_v5['params_m']
    
    # Вывод сравнительной таблицы
    print("\n" + "="*80)
    print("COMPARISON TABLE")
    print("="*80)
    print(f"{'Metric':<25} {'YOLOX':<15} {'YOLOv5':<15} {'Diff':<15}")
    print("-"*80)
    
    # mAP
    map_x = results['yolox']['mAP@0.5']
    map_v5 = results['yolov5']['mAP@0.5']
    diff_map = (map_v5 - map_x) * 100
    print(f"{'mAP@0.5 (%)':<25} {map_x*100:<15.2f} {map_v5*100:<15.2f} {diff_map:>+14.2f}%")
    
    # FPS
    fps_x = results['yolox']['fps']
    fps_v5 = results['yolov5']['fps']
    diff_fps = ((fps_v5 - fps_x) / fps_x * 100) if fps_x > 0 else 0
    print(f"{'FPS':<25} {fps_x:<15.2f} {fps_v5:<15.2f} {diff_fps:>+14.1f}%")
    
    # Latency
    lat_x = results['yolox']['avg_inference_time_ms']
    lat_v5 = results['yolov5']['avg_inference_time_ms']
    diff_lat = ((lat_x - lat_v5) / lat_x * 100) if lat_x > 0 else 0
    print(f"{'Latency (ms)':<25} {lat_x:<15.2f} {lat_v5:<15.2f} {diff_lat:>+14.1f}%")
    
    # FLOPS
    flop_x = results['yolox']['flops_g']
    flop_v5 = results['yolov5']['flops_g']
    print(f"{'FLOPS (G)':<25} {flop_x:<15.2f} {flop_v5:<15.2f} {'-':<15}")
    
    # Params
    par_x = results['yolox']['params_m']
    par_v5 = results['yolov5']['params_m']
    print(f"{'Params (M)':<25} {par_x:<15.2f} {par_v5:<15.2f} {'-':<15}")
    
    print("="*80)
    
    # Per-class AP comparison
    print("\n" + "="*80)
    print("PER-CLASS AP COMPARISON")
    print("="*80)
    print(f"{'Class':<15} {'YOLOX':<12} {'YOLOv5':<12} {'Δ':<12}")
    print("-"*80)
    
    ap_x = results['yolox']['per_class_ap']
    ap_v5 = results['yolov5']['per_class_ap']
    
    for class_name in VOC_NAMES:
        v1 = ap_x.get(class_name, 0) * 100
        v2 = ap_v5.get(class_name, 0) * 100
        delta = v2 - v1
        print(f"{class_name:<15} {v1:<12.2f} {v2:<12.2f} {delta:>+11.2f}%")
    
    print("="*80)
    
    # Сохраняем JSON
    with open(output_json, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {output_json}")
    
    return results


# ============================================
# Пример использования
# ============================================
if __name__ == "__main__":
    # Конфигурация путей
    VOC_ROOT = "data/VOC2012"  # Путь к VOC (должны быть JPEGImages/, Annotations/, ImageSets/)
    YOLOX_MODEL = "models/yolox_tiny.onnx"
    YOLOV5_MODEL = "models/yolov5nu.onnx"
    
    # Режимы: 'eval', 'compare', 'single'
    MODE = 'compare'
    
    if MODE == 'compare':
        # Полное сравнение двух моделей на VOC
        compare_models_on_voc(
            voc_root=VOC_ROOT,
            yolox_path=YOLOX_MODEL,
            yolov5_path=YOLOV5_MODEL,
            image_set='val',  # или 'train', 'test'
            voc_year='2012'   # или '2007'
        )
    
    elif MODE == 'eval':
        # Оценка одной модели
        detector = YOLODetector(YOLOX_MODEL, model_type='yolox', conf_thres=0.01)
        results = detector.evaluate_voc(VOC_ROOT, image_set='val', voc_year='2012')
        
        # Дополнительно: бенчмарк чистой скорости
        print("\n[SPEED BENCHMARK]")
        speed = detector.benchmark_speed(num_runs=100)
        print(f"Pure inference FPS: {speed['fps']:.2f}")
    
    elif MODE == 'single':
        # Тест на одном изображении с показом VOC-классов
        detector = YOLODetector(YOLOX_MODEL, model_type='yolox', conf_thres=0.3)
        
        test_img = f"{VOC_ROOT}/JPEGImages/2007_000032.jpg"  # пример
        detections, img = detector.predict(test_img, filter_voc_only=True)
        
        print(f"\nDetected {len(detections)} VOC objects:")
        for det in detections:
            x1, y1, x2, y2, score, coco_cls = det
            voc_cls = COCO_TO_VOC_INDEX.get(int(coco_cls), 'unknown')
            voc_name = VOC_NAMES[voc_cls] if voc_cls != 'unknown' else 'unknown'
            print(f"  {voc_name}: {score:.3f} [{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]")


PASCAL VOC EVALUATION: YOLOX vs YOLOv5
Dataset: VOC 2012 val
COCO->VOC mapping: 20 classes

[1/2] Evaluating YOLOX...


2026-03-26 16:23:10.718 Python[52729:27509284] 2026-03-26 16:23:10.717339 [W:onnxruntime:, coreml_execution_provider.cc:113 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 2 number of nodes in the graph: 277 number of nodes supported by CoreML: 276
2026-03-26 16:23:11.907 Python[52729:27509284] 2026-03-26 16:23:11.907198 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-03-26 16:23:11.907 Python[52729:27509284] 2026-03-26 16:23:11.907223 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.



VOC 2012 val Evaluation
Всего изображений: 5823
Модель: YOLOX
IoU threshold: 0.5

Processing [0/5823] 2008_000002...
Processing [100/5823] 2008_000406...
Processing [200/5823] 2008_000804...
Processing [300/5823] 2008_001183...
Processing [400/5823] 2008_001549...
Processing [500/5823] 2008_001914...
Processing [600/5823] 2008_002312...
Processing [700/5823] 2008_002696...
Processing [800/5823] 2008_003089...
Processing [900/5823] 2008_003492...
Processing [1000/5823] 2008_003873...
Processing [1100/5823] 2008_004263...
Processing [1200/5823] 2008_004615...
Processing [1300/5823] 2008_005010...
Processing [1400/5823] 2008_005417...
Processing [1500/5823] 2008_005808...
Processing [1600/5823] 2008_006190...
Processing [1700/5823] 2008_006662...
Processing [1800/5823] 2008_007056...
Processing [1900/5823] 2008_007455...
Processing [2000/5823] 2008_007841...
Processing [2100/5823] 2008_008272...
Processing [2200/5823] 2008_008684...
Processing [2300/5823] 2009_000300...
Processing [2400/

2026-03-26 16:24:02.416 Python[52729:27509284] 2026-03-26 16:24:02.416085 [W:onnxruntime:, coreml_execution_provider.cc:113 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 3 number of nodes in the graph: 262 number of nodes supported by CoreML: 258
2026-03-26 16:24:03.091 Python[52729:27509284] 2026-03-26 16:24:03.091322 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-03-26 16:24:03.091 Python[52729:27509284] 2026-03-26 16:24:03.091344 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


[INFO] YOLOv5 формат выхода: single

VOC 2012 val Evaluation
Всего изображений: 5823
Модель: YOLOV5
IoU threshold: 0.5

Processing [0/5823] 2008_000002...
Processing [100/5823] 2008_000406...
Processing [200/5823] 2008_000804...
Processing [300/5823] 2008_001183...
Processing [400/5823] 2008_001549...
Processing [500/5823] 2008_001914...
Processing [600/5823] 2008_002312...
Processing [700/5823] 2008_002696...
Processing [800/5823] 2008_003089...
Processing [900/5823] 2008_003492...
Processing [1000/5823] 2008_003873...
Processing [1100/5823] 2008_004263...
Processing [1200/5823] 2008_004615...
Processing [1300/5823] 2008_005010...
Processing [1400/5823] 2008_005417...
Processing [1500/5823] 2008_005808...
Processing [1600/5823] 2008_006190...
Processing [1700/5823] 2008_006662...
Processing [1800/5823] 2008_007056...
Processing [1900/5823] 2008_007455...
Processing [2000/5823] 2008_007841...
Processing [2100/5823] 2008_008272...
Processing [2200/5823] 2008_008684...
Processing [2300/5